In [ ]:
content = content = """
... # TECHNOVA SOLUTIONS: EMPLOYEE POLICY HANDBOOK

## 1. LEAVE POLICY
At TechNova Solutions, we value work-life balance.
- Annual Leave: Full-time employees are entitled to 20 days of paid annual leave per calendar year, accrued monthly.
- Sick Leave: Employees receive 10 days of paid sick leave annually for personal illness or family medical care.
- Parental Leave: We offer 12 weeks of fully paid parental leave for new parents (birth or adoption).
- Bereavement Leave: 3 days of paid leave are provided in the event of the death of an immediate family member.
- Approval Process: All leave requests must be submitted via the HR portal at least two weeks in advance.
- Unused Leave: Up to 5 days of annual leave can be carried over to the next calendar year.

## 2. REMOTE WORK POLICY
- Eligibility: Employees may work remotely up to 3 days per week with manager approval.
- Home Office: A one-time home-office stipend of $500 is provided to all new hires for desk, chair, or monitor setup.
- Core Hours: Core hours are 10:00 AM to 3:00 PM EST, during which all employees must be available for meetings.
- Security: Remote workers must use a company-provided VPN and multi-factor authentication (MFA) at all times.

## 3. CODE OF CONDUCT
- Zero Tolerance: TechNova maintains a zero-tolerance policy for harassment, discrimination, or bullying.
- Confidentiality: Employees must not disclose proprietary software code or client data to external parties.
- Professionalism: Employees are expected to maintain a professional demeanor in all communications.
- Dress Code: Business casual is required for office days and client video calls.

## 4. EXPENSE REIMBURSEMENT
- Travel: Travel expenses for client meetings are fully reimbursable.
- Internet: Monthly internet stipends of $50 are provided for remote-eligible roles.
- Submission: All receipts must be submitted via the Expense tool within 30 days of the expense.

## 5. HEALTH AND WELLNESS
- Health Insurance: Comprehensive medical, dental, and vision insurance starts on the first day of employment.
- Gym Membership: TechNova reimburses up to $60 per month for fitness center memberships or yoga classes.
- Mental Health: Employees have access to an Employee Assistance Program (EAP) offering 5 free counseling sessions per year.

## 6. SOCIAL MEDIA POLICY
- Personal Use: Employees may not post content representing TechNova without prior Marketing team approval.
- Prohibited Content: Sharing internal project screenshots or code snippets on social media is strictly prohibited.
- Disclaimer: If mentioning TechNova in a personal bio, employees should state: "Opinions are my own."

## 7. PERFORMANCE AND PROMOTIONS
- Reviews: Performance reviews occur bi-annually in June and December.
- Promotions: Promotion cycles are handled annually. Eligibility requires a "Meets Expectations" rating for two consecutive cycles.
- Training: TechNova provides an annual $2,000 learning budget for certifications, books, or conferences.
 ...
"""



In [ ]:
with open("company_policy.txt", "w") as f:
    f.write(content.strip())

print("File 'company_policy.txt' created successfully.")

File 'company_policy.txt' created successfully.


In [ ]:
!pip install langchain langchain-community langchain-google-genai chromadb google-generativeai

INFO: pip is looking at multiple versions of langchain-google-genai to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of langchain-google-genai to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of opentelemetry-proto to determine which version is compatible with other requirements. This could take a while.
  Using cached opentelemetry_exporter_otlp_proto_common-1.40.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.40.0-py3-none-any.whl.metadata (2.6 kB)
INFO: pip is still looking at multiple versions of opentelemetry-proto to determine 

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma, FAISS

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
from langchain_core.runnables import RunnablePassthrough
import os

In [ ]:
os.environ["GOOGLE_API_KEY"] = "AIzaSyD7Z6KBAril9wgsugdggs79hdbfdhgRPBqmm3Ro2Wn7aB27k"

In [ ]:
loader = TextLoader("company_policy.txt")
documents = loader.load()

In [ ]:
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=0)
chunks = text_splitter.split_documents(documents)

In [ ]:
import google.generativeai as genai
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print(m.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview


In [ ]:
!pip install -q --upgrade chromadb

In [ ]:
import numpy as np

embeddings_model = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")


texts = [doc.page_content for doc in chunks]
metadatas = [doc.metadata for doc in chunks]


raw_embeddings_from_model = embeddings_model.embed_documents(texts)


pre_computed_embeddings = [np.array(vec, dtype=np.float32).tolist() for vec in raw_embeddings_from_model]

faiss_embedding_data = list(zip(texts, pre_computed_embeddings))

vector_db = FAISS.from_embeddings(text_embeddings=faiss_embedding_data, embedding=embeddings_model, metadatas=metadatas)

In [ ]:
!pip install faiss-cpu

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [ ]:
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
""")

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
rag_chain = (
    {"context": vector_db.as_retriever() | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
query = "performance and promotions?"

In [ ]:
response = rag_chain.invoke(query)
print(f"Gemini Answer: {response}")

Gemini Answer: **Performance and Promotions:**

*   **Reviews:** Performance reviews occur bi-annually in June and December.
*   **Promotions:** Promotion cycles are handled annually. Eligibility requires a "Meets Expectations" rating for two consecutive cycles.
*   **Training:** TechNova provides an annual $2,000 learning budget for certifications, books, or conferences.
